In [1]:
# ============================================================
# EXPERIMENT A
# RANDOM STRATIFIED SPLIT
# 20 MI FEATURES + RANDOM FOREST
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

RANDOM_SEED = 2024
TEST_SIZE = 0.20
N_ESTIMATORS = 100

FINAL_DATA_FILE = "CIC-IDS2017_20features_clean.csv"

selected_features = [
    "Avg Packet Size",
    "Bwd Packet Length Mean",
    "Avg Bwd Segment Size",
    "Subflow Bwd Bytes",
    "Bwd Packets Length Total",
    "Bwd Packet Length Max",
    "Packet Length Std",
    "Packet Length Variance",
    "Fwd IAT Max",
    "Packet Length Mean",
    "Fwd IAT Total",
    "Fwd Packets Length Total",
    "Subflow Fwd Bytes",
    "Packet Length Max",
    "Bwd Packet Length Std",
    "Init Bwd Win Bytes",
    "Fwd Packet Length Max",
    "Init Fwd Win Bytes",
    "Fwd IAT Mean",
    "Fwd Packet Length Std"
]

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(FINAL_DATA_FILE)

X = df[selected_features].copy()
y = df["Label"].copy()

X = X.astype(np.float32)
y = y.astype(np.int8)

print("Dataset shape:", X.shape)
print("Label distribution:")
print(y.value_counts())

# ------------------------------------------------------------
# RANDOM STRATIFIED SPLIT
# ------------------------------------------------------------

X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

print("\n===== EXPERIMENT A SPLIT =====")
print("Train:", X_train_A.shape)
print("Test :", X_test_A.shape)

print("\nTrain distribution:")
print(y_train_A.value_counts())

print("\nTest distribution:")
print(y_test_A.value_counts())

# ------------------------------------------------------------
# RANDOM FOREST
# ------------------------------------------------------------

rf_A = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_SEED,
    n_jobs=1,
    class_weight=None
)

start_time = time.time()

rf_A.fit(X_train_A, y_train_A)

training_time_A = time.time() - start_time

# ------------------------------------------------------------
# PREDICTION
# ------------------------------------------------------------

y_pred_A = rf_A.predict(X_test_A)
y_prob_A = rf_A.predict_proba(X_test_A)[:, 1]

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

results_A = {
    "Experiment": "A - Random Stratified Split",
    "Model": "Random Forest",
    "Handling": "None",
    "Seed": RANDOM_SEED,
    "Accuracy": accuracy_score(y_test_A, y_pred_A),
    "Precision": precision_score(y_test_A, y_pred_A),
    "Recall": recall_score(y_test_A, y_pred_A),
    "F1": f1_score(y_test_A, y_pred_A),
    "ROC_AUC": roc_auc_score(y_test_A, y_prob_A),
    "PR_AUC": average_precision_score(y_test_A, y_prob_A),
    "Training_Time_sec": training_time_A
}

results_A_df = pd.DataFrame([results_A])

print("\n===== EXPERIMENT A RESULTS =====")
print(results_A_df.to_string(index=False))

Dataset shape: (1265086, 20)
Label distribution:
Label
0    943316
1    321770
Name: count, dtype: int64

===== EXPERIMENT A SPLIT =====
Train: (1012068, 20)
Test : (253018, 20)

Train distribution:
Label
0    754652
1    257416
Name: count, dtype: int64

Test distribution:
Label
0    188664
1     64354
Name: count, dtype: int64

===== EXPERIMENT A RESULTS =====
                 Experiment         Model Handling  Seed  Accuracy  Precision   Recall       F1  ROC_AUC   PR_AUC  Training_Time_sec
A - Random Stratified Split Random Forest     None  2024  0.999526   0.999192 0.998943 0.999068 0.999932 0.999883         160.119804


In [2]:
# ============================================================
# EXPERIMENT B
# DUPLICATE-AWARE / GROUPED SPLIT
# 20 MI FEATURES + RANDOM FOREST
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------
# CREATE GROUP ID FROM 20-FEATURE VECTOR
# ------------------------------------------------------------

# Setiap feature vector identik mendapatkan group yang sama
groups = pd.util.hash_pandas_object(
    X,
    index=False
)

print("Total samples :", len(X))
print("Unique groups :", groups.nunique())

# ------------------------------------------------------------
# GROUP-AWARE 80/20 SPLIT
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED
)

train_idx_B, test_idx_B = next(
    gss.split(X, y, groups=groups)
)

X_train_B = X.iloc[train_idx_B].copy()
X_test_B  = X.iloc[test_idx_B].copy()

y_train_B = y.iloc[train_idx_B].copy()
y_test_B  = y.iloc[test_idx_B].copy()

print("\n===== EXPERIMENT B SPLIT =====")
print("Train:", X_train_B.shape)
print("Test :", X_test_B.shape)

print("\nTrain distribution:")
print(y_train_B.value_counts())

print("\nTest distribution:")
print(y_test_B.value_counts())

# ------------------------------------------------------------
# CHECK GROUP OVERLAP
# ------------------------------------------------------------

train_groups_B = set(groups.iloc[train_idx_B])
test_groups_B = set(groups.iloc[test_idx_B])

group_overlap_B = train_groups_B.intersection(test_groups_B)

print("\n===== GROUP OVERLAP CHECK =====")
print("Train groups:", len(train_groups_B))
print("Test groups :", len(test_groups_B))
print("Shared groups:", len(group_overlap_B))

assert len(group_overlap_B) == 0, \
    "ERROR: masih terdapat group yang overlap!"

print("✓ Tidak ada feature-vector group yang terbagi antara train dan test.")

# ------------------------------------------------------------
# RANDOM FOREST
# ------------------------------------------------------------

rf_B = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_SEED,
    n_jobs=1,
    class_weight=None
)

start_time = time.time()

rf_B.fit(X_train_B, y_train_B)

training_time_B = time.time() - start_time

# ------------------------------------------------------------
# PREDICTION
# ------------------------------------------------------------

y_pred_B = rf_B.predict(X_test_B)
y_prob_B = rf_B.predict_proba(X_test_B)[:, 1]

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

results_B = {
    "Experiment": "B - Duplicate-Aware Split",
    "Model": "Random Forest",
    "Handling": "None",
    "Seed": RANDOM_SEED,
    "Accuracy": accuracy_score(y_test_B, y_pred_B),
    "Precision": precision_score(y_test_B, y_pred_B),
    "Recall": recall_score(y_test_B, y_pred_B),
    "F1": f1_score(y_test_B, y_pred_B),
    "ROC_AUC": roc_auc_score(y_test_B, y_prob_B),
    "PR_AUC": average_precision_score(y_test_B, y_prob_B),
    "Training_Time_sec": training_time_B
}

results_B_df = pd.DataFrame([results_B])

print("\n===== EXPERIMENT B RESULTS =====")
print(results_B_df.to_string(index=False))

Total samples : 1265086
Unique groups : 786885

===== EXPERIMENT B SPLIT =====
Train: (1015308, 20)
Test : (249778, 20)

Train distribution:
Label
0    757558
1    257750
Name: count, dtype: int64

Test distribution:
Label
0    185758
1     64020
Name: count, dtype: int64

===== GROUP OVERLAP CHECK =====
Train groups: 629508
Test groups : 157377
Shared groups: 0
✓ Tidak ada feature-vector group yang terbagi antara train dan test.

===== EXPERIMENT B RESULTS =====
               Experiment         Model Handling  Seed  Accuracy  Precision   Recall       F1  ROC_AUC   PR_AUC  Training_Time_sec
B - Duplicate-Aware Split Random Forest     None  2024  0.998967   0.999389 0.996579 0.997982 0.999916 0.999861         137.023198


In [3]:
# ============================================================
# COMPARISON A vs B
# ============================================================

comparison = pd.concat(
    [results_A_df, results_B_df],
    ignore_index=True
)

print("\n===== EXPERIMENT A vs B =====")

print(
    comparison[
        [
            "Experiment",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC_AUC",
            "PR_AUC",
            "Training_Time_sec"
        ]
    ].to_string(index=False)
)


===== EXPERIMENT A vs B =====
                 Experiment  Accuracy  Precision   Recall       F1  ROC_AUC   PR_AUC  Training_Time_sec
A - Random Stratified Split  0.999526   0.999192 0.998943 0.999068 0.999932 0.999883         160.119804
  B - Duplicate-Aware Split  0.998967   0.999389 0.996579 0.997982 0.999916 0.999861         137.023198


In [4]:
# ============================================================
# EXPERIMENT C
# DUPLICATE-AWARE + STRATIFIED SPLIT
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from sklearn.model_selection import StratifiedGroupKFold

In [5]:
# ============================================================
# LOAD DATA
# ============================================================

RANDOM_SEED = 2024
TEST_SIZE = 0.20
N_ESTIMATORS = 100

FINAL_DATA_FILE = "CIC-IDS2017_20features_clean.csv"

selected_features = [
    "Avg Packet Size",
    "Bwd Packet Length Mean",
    "Avg Bwd Segment Size",
    "Subflow Bwd Bytes",
    "Bwd Packets Length Total",
    "Bwd Packet Length Max",
    "Packet Length Std",
    "Packet Length Variance",
    "Fwd IAT Max",
    "Packet Length Mean",
    "Fwd IAT Total",
    "Fwd Packets Length Total",
    "Subflow Fwd Bytes",
    "Packet Length Max",
    "Bwd Packet Length Std",
    "Init Bwd Win Bytes",
    "Fwd Packet Length Max",
    "Init Fwd Win Bytes",
    "Fwd IAT Mean",
    "Fwd Packet Length Std"
]

df_C = pd.read_csv(FINAL_DATA_FILE)

X_C = df_C[selected_features].copy()
y_C = df_C["Label"].copy()

X_C = X_C.astype(np.float32)
y_C = y_C.astype(np.int8)

print("===== DATASET =====")
print("X shape:", X_C.shape)
print("y shape:", y_C.shape)

print("\nLabel distribution:")
print(y_C.value_counts())

print("\nLabel proportion:")
print(y_C.value_counts(normalize=True))

===== DATASET =====
X shape: (1265086, 20)
y shape: (1265086,)

Label distribution:
Label
0    943316
1    321770
Name: count, dtype: int64

Label proportion:
Label
0    0.745654
1    0.254346
Name: proportion, dtype: float64


In [6]:
# ============================================================
# CREATE GROUPS FROM IDENTICAL 20-FEATURE VECTORS
# ============================================================

groups_C = pd.factorize(
    pd.util.hash_pandas_object(
        X_C,
        index=False
    )
)[0]

groups_C = pd.Series(groups_C, index=X_C.index)

print("\n===== GROUP INFORMATION =====")
print("Total samples :", len(X_C))
print("Unique groups :", groups_C.nunique())
print(
    "Samples in repeated groups:",
    len(X_C) - groups_C.nunique()
)


===== GROUP INFORMATION =====
Total samples : 1265086
Unique groups : 786885
Samples in repeated groups: 478201


In [7]:
# ============================================================
# DUPLICATE-AWARE + STRATIFIED SPLIT
# ============================================================

N_SPLITS = 5

sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

best_split = None
best_difference = float("inf")

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(X_C, y_C, groups_C)
):

    test_ratio = len(test_idx) / len(X_C)

    difference = abs(test_ratio - TEST_SIZE)

    print(
        f"Fold {fold}: "
        f"train={len(train_idx)}, "
        f"test={len(test_idx)}, "
        f"test_ratio={test_ratio:.4f}"
    )

    if difference < best_difference:
        best_difference = difference
        best_split = (train_idx, test_idx)

train_idx_C, test_idx_C = best_split

print("\n===== SELECTED SPLIT =====")
print("Train samples:", len(train_idx_C))
print("Test samples :", len(test_idx_C))
print(
    "Test ratio   :",
    round(len(test_idx_C) / len(X_C) * 100, 2),
    "%"
)

Fold 0: train=1004630, test=260456, test_ratio=0.2059
Fold 1: train=1008607, test=256479, test_ratio=0.2027
Fold 2: train=1013738, test=251348, test_ratio=0.1987
Fold 3: train=1014030, test=251056, test_ratio=0.1984
Fold 4: train=1019339, test=245747, test_ratio=0.1943

===== SELECTED SPLIT =====
Train samples: 1013738
Test samples : 251348
Test ratio   : 19.87 %


In [8]:
# ============================================================
# CREATE TRAIN / TEST DATA
# ============================================================

X_train_C = X_C.iloc[train_idx_C].copy()
X_test_C = X_C.iloc[test_idx_C].copy()

y_train_C = y_C.iloc[train_idx_C].copy()
y_test_C = y_C.iloc[test_idx_C].copy()

groups_train_C = groups_C.iloc[train_idx_C]
groups_test_C = groups_C.iloc[test_idx_C]

print("\n===== TRAIN / TEST =====")
print("X_train:", X_train_C.shape)
print("X_test :", X_test_C.shape)

print("\nTrain label distribution:")
print(y_train_C.value_counts())

print("\nTrain label proportion:")
print(
    y_train_C.value_counts(normalize=True)
)

print("\nTest label distribution:")
print(y_test_C.value_counts())

print("\nTest label proportion:")
print(
    y_test_C.value_counts(normalize=True)
)


===== TRAIN / TEST =====
X_train: (1013738, 20)
X_test : (251348, 20)

Train label distribution:
Label
0    758290
1    255448
Name: count, dtype: int64

Train label proportion:
Label
0    0.748014
1    0.251986
Name: proportion, dtype: float64

Test label distribution:
Label
0    185026
1     66322
Name: count, dtype: int64

Test label proportion:
Label
0    0.736135
1    0.263865
Name: proportion, dtype: float64


In [9]:
# ============================================================
# CHECK GROUP OVERLAP
# ============================================================

train_groups_C = set(groups_train_C)
test_groups_C = set(groups_test_C)

shared_groups_C = train_groups_C.intersection(
    test_groups_C
)

print("\n===== GROUP OVERLAP CHECK =====")
print("Train unique groups :", len(train_groups_C))
print("Test unique groups  :", len(test_groups_C))
print("Shared groups       :", len(shared_groups_C))

if len(shared_groups_C) == 0:
    print(
        "✓ PASS: Tidak ada identical feature-vector "
        "group yang terbagi antara train dan test."
    )
else:
    print(
        "✗ FAIL: Masih terdapat group yang overlap."
    )

assert len(shared_groups_C) == 0


===== GROUP OVERLAP CHECK =====
Train unique groups : 629471
Test unique groups  : 157414
Shared groups       : 0
✓ PASS: Tidak ada identical feature-vector group yang terbagi antara train dan test.


In [10]:
# ============================================================
# CLASS DISTRIBUTION CHECK
# ============================================================

overall_ratio = y_C.mean()
train_ratio = y_train_C.mean()
test_ratio = y_test_C.mean()

print("\n===== CLASS DISTRIBUTION CHECK =====")

print(
    f"Overall ATTACK ratio : {overall_ratio:.6f}"
)

print(
    f"Train ATTACK ratio   : {train_ratio:.6f}"
)

print(
    f"Test ATTACK ratio    : {test_ratio:.6f}"
)

print(
    f"Train difference     : "
    f"{abs(train_ratio - overall_ratio):.6f}"
)

print(
    f"Test difference      : "
    f"{abs(test_ratio - overall_ratio):.6f}"
)


===== CLASS DISTRIBUTION CHECK =====
Overall ATTACK ratio : 0.254346
Train ATTACK ratio   : 0.251986
Test ATTACK ratio    : 0.263865
Train difference     : 0.002360
Test difference      : 0.009519


In [11]:
# ============================================================
# RANDOM FOREST
# EXPERIMENT C
# ============================================================

rf_C = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_SEED,
    n_jobs=1,
    class_weight=None
)

print("\n===== TRAINING RANDOM FOREST =====")

start_time_C = time.time()

rf_C.fit(
    X_train_C,
    y_train_C
)

training_time_C = time.time() - start_time_C

print(
    f"Training time: "
    f"{training_time_C:.2f} seconds"
)


===== TRAINING RANDOM FOREST =====
Training time: 136.16 seconds


In [12]:
# ============================================================
# EVALUATION
# ============================================================

y_pred_C = rf_C.predict(X_test_C)

y_prob_C = rf_C.predict_proba(
    X_test_C
)[:, 1]

accuracy_C = accuracy_score(
    y_test_C,
    y_pred_C
)

precision_C = precision_score(
    y_test_C,
    y_pred_C
)

recall_C = recall_score(
    y_test_C,
    y_pred_C
)

f1_C = f1_score(
    y_test_C,
    y_pred_C
)

roc_auc_C = roc_auc_score(
    y_test_C,
    y_prob_C
)

pr_auc_C = average_precision_score(
    y_test_C,
    y_prob_C
)

print("\n===== EXPERIMENT C RESULTS =====")

print(f"Accuracy : {accuracy_C:.6f}")
print(f"Precision: {precision_C:.6f}")
print(f"Recall   : {recall_C:.6f}")
print(f"F1       : {f1_C:.6f}")
print(f"ROC-AUC  : {roc_auc_C:.6f}")
print(f"PR-AUC   : {pr_auc_C:.6f}")
print(f"Time     : {training_time_C:.2f} sec")


===== EXPERIMENT C RESULTS =====
Accuracy : 0.998950
Precision: 0.998160
Recall   : 0.997859
F1       : 0.998009
ROC-AUC  : 0.999954
PR-AUC   : 0.999888
Time     : 136.16 sec


In [13]:
# ============================================================
# SAVE EXPERIMENT C RESULT
# ============================================================

results_C = pd.DataFrame([{
    "Experiment": "C - Duplicate-Aware + Stratified",
    "Model": "Random Forest",
    "Handling": "None",
    "Seed": RANDOM_SEED,
    "Train_Samples": len(X_train_C),
    "Test_Samples": len(X_test_C),
    "Accuracy": accuracy_C,
    "Precision": precision_C,
    "Recall": recall_C,
    "F1": f1_C,
    "ROC_AUC": roc_auc_C,
    "PR_AUC": pr_auc_C,
    "Training_Time_sec": training_time_C
}])

print("\n===== RESULT TABLE =====")
display(results_C)


===== RESULT TABLE =====


,Experiment,Model,Handling,Seed,Train_Samples,Test_Samples,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Training_Time_sec
0,C - Duplicate-Aware + Stratified,Random Forest,None,2024,1013738,251348,0.99895,0.99816,0.997859,0.998009,0.999954,0.999888,136.155035


In [14]:
# ============================================================
# COMPARE A vs B vs C
# ============================================================

comparison_ABC = pd.concat(
    [
        results_A_df,
        results_B_df,
        results_C
    ],
    ignore_index=True
)

print("\n===== EXPERIMENT A vs B vs C =====")

display(
    comparison_ABC[
        [
            "Experiment",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC_AUC",
            "PR_AUC",
            "Training_Time_sec"
        ]
    ]
)


===== EXPERIMENT A vs B vs C =====


,Experiment,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Training_Time_sec
0,A - Random Stratified Split,0.999526,0.999192,0.998943,0.999068,0.999932,0.999883,160.119804
1,B - Duplicate-Aware Split,0.998967,0.999389,0.996579,0.997982,0.999916,0.999861,137.023198
2,C - Duplicate-Aware + Stratified,0.998950,0.998160,0.997859,0.998009,0.999954,0.999888,136.155035
